# ECG Project — Notebook 5
## Hybrid Five-Level Risk Model Training

This notebook trains the final ECG risk-screening system.

### Final design

- **Levels 1–4:** machine-learning classifier
- **Level 5:** high-priority critical-rule layer
- **Output:** five probabilities, predicted risk level, possible ECG findings, and prediction reasons

> This project produces ECG-based screening support only. It is not a confirmed diagnosis, emergency decision system, or replacement for clinical assessment.


## Why use a hybrid model?

Only a few reports belong to Level 5. A supervised model cannot reliably learn a separate class from such a small sample.

Therefore:

1. A classifier learns Levels 1–4.
2. Critical ECG patterns are checked first.
3. When a critical pattern is detected, the final result becomes Level 5.
4. Otherwise, the calibrated machine-learning probabilities are used.


## 1. Install packages

In [ ]:
!pip install -q pandas numpy scikit-learn joblib openpyxl matplotlib

import io
import json
import re
import shutil
import zipfile
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)


## 2. Upload `ecg_risk_label_outputs.zip` from Notebook 4

In [ ]:
from google.colab import files

uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise ValueError(
        "Upload ecg_risk_label_outputs.zip created by Notebook 4."
    )

input_zip = Path(zip_names[0])
print("Selected:", input_zip)


## 3. Extract and load the fixed datasets

In [ ]:
input_dir = Path("/content/notebook4_outputs")

if input_dir.exists():
    shutil.rmtree(input_dir)

input_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(input_zip, "r") as archive:
    archive.extractall(input_dir)

required = [
    "train_risk.csv",
    "validation_risk.csv",
    "test_risk.csv",
    "manual_review_risk_reports.csv",
    "risk_labeled_dataset.csv"
]

missing = [
    filename for filename in required
    if not (input_dir / filename).exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing required files: {missing}"
    )

train_df = pd.read_csv(input_dir / "train_risk.csv")
validation_df = pd.read_csv(input_dir / "validation_risk.csv")
test_df = pd.read_csv(input_dir / "test_risk.csv")
manual_review_df = pd.read_csv(
    input_dir / "manual_review_risk_reports.csv"
)
full_labeled_df = pd.read_csv(
    input_dir / "risk_labeled_dataset.csv"
)

print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))
print("Manual review rows:", len(manual_review_df))

display(
    pd.DataFrame({
        "train": train_df["risk_level"].value_counts().sort_index(),
        "validation": validation_df["risk_level"].value_counts().sort_index(),
        "test": test_df["risk_level"].value_counts().sort_index()
    }).fillna(0).astype(int)
)


## 4. Define model features

To avoid label leakage, the model does **not** use:

- `risk_level`
- `risk_name`
- `risk_score`
- `risk_reasons`
- rule-preview probabilities
- standardized finding labels

The classifier uses extracted ECG measurements plus selected raw report information.

Because the risk labels were generated partly from these measurements and machine findings, this is a **weakly supervised model** that learns to reproduce the provisional labeling logic.


In [ ]:
NUMERIC_FEATURES = [
    "heart_rate_bpm",
    "p_duration_ms",
    "pr_interval_ms",
    "qrs_duration_ms",
    "qt_interval_ms",
    "qtc_bazett_ms",
    "p_axis_deg",
    "qrs_axis_deg",
    "t_axis_deg",
    "rv5_mv",
    "sv1_mv"
]

CATEGORICAL_FEATURES = [
    "rhythm",
    "primary_label"
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing_features = [
    feature for feature in ALL_FEATURES
    if feature not in train_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing feature columns: {missing_features}"
    )

for dataframe in [train_df, validation_df, test_df]:
    dataframe[CATEGORICAL_FEATURES] = (
        dataframe[CATEGORICAL_FEATURES]
        .fillna("Unknown")
        .astype(str)
    )

print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)


## 5. Separate Level 5 from machine-learning training

The machine-learning classifier is trained only on Levels 1–4.

Level-5 records remain available for evaluating the critical-rule layer.


In [ ]:
def split_ml_and_critical(df):
    ml_part = df.loc[
        df["risk_level"].isin([1, 2, 3, 4])
    ].copy()

    critical_part = df.loc[
        df["risk_level"] == 5
    ].copy()

    return ml_part, critical_part

train_ml, train_level5 = split_ml_and_critical(train_df)
validation_ml, validation_level5 = split_ml_and_critical(validation_df)
test_ml, test_level5 = split_ml_and_critical(test_df)

print("ML train rows:", len(train_ml))
print("ML validation rows:", len(validation_ml))
print("ML test rows:", len(test_ml))

print("Level-5 train rows:", len(train_level5))
print("Level-5 validation rows:", len(validation_level5))
print("Level-5 test rows:", len(test_level5))


## 6. Build preprocessing pipelines

In [ ]:
numeric_transformer_scaled = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

numeric_transformer_tree = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor_scaled = ColumnTransformer([
    (
        "numeric",
        numeric_transformer_scaled,
        NUMERIC_FEATURES
    ),
    (
        "categorical",
        categorical_transformer,
        CATEGORICAL_FEATURES
    )
])

preprocessor_tree = ColumnTransformer([
    (
        "numeric",
        numeric_transformer_tree,
        NUMERIC_FEATURES
    ),
    (
        "categorical",
        categorical_transformer,
        CATEGORICAL_FEATURES
    )
])


## 7. Prepare training data

In [ ]:
X_train = train_ml[ALL_FEATURES]
y_train = train_ml["risk_level"].astype(int)

X_validation = validation_ml[ALL_FEATURES]
y_validation = validation_ml["risk_level"].astype(int)

X_test = test_ml[ALL_FEATURES]
y_test = test_ml["risk_level"].astype(int)

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)


## 8. Define candidate models

Models compared:

- Logistic Regression
- Random Forest
- Extra Trees
- HistGradientBoosting

Model selection uses:

1. Macro F1-score
2. Balanced accuracy
3. Validation log loss


In [ ]:
candidate_models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42,
                multi_class="auto"
            )
        )
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_tree),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=2
            )
        )
    ]),

    "Extra Trees": Pipeline([
        ("preprocessor", preprocessor_tree),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=500,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=2
            )
        )
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", preprocessor_tree),
        (
            "classifier",
            HistGradientBoostingClassifier(
                learning_rate=0.06,
                max_iter=300,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42
            )
        )
    ])
}


## 9. Train and compare models on the validation set

In [ ]:
model_results = []
fitted_models = {}

for name, model in candidate_models.items():
    print(f"Training {name}...")

    fitted = clone(model)
    fitted.fit(X_train, y_train)

    prediction = fitted.predict(X_validation)
    probability = fitted.predict_proba(X_validation)

    macro_f1 = f1_score(
        y_validation,
        prediction,
        average="macro",
        zero_division=0
    )

    balanced_acc = balanced_accuracy_score(
        y_validation,
        prediction
    )

    accuracy = accuracy_score(
        y_validation,
        prediction
    )

    validation_log_loss = log_loss(
        y_validation,
        probability,
        labels=[1, 2, 3, 4]
    )

    try:
        auc_macro = roc_auc_score(
            y_validation,
            probability,
            multi_class="ovr",
            average="macro",
            labels=[1, 2, 3, 4]
        )
    except Exception:
        auc_macro = np.nan

    model_results.append({
        "model": name,
        "validation_accuracy": accuracy,
        "validation_balanced_accuracy": balanced_acc,
        "validation_macro_f1": macro_f1,
        "validation_log_loss": validation_log_loss,
        "validation_macro_roc_auc": auc_macro
    })

    fitted_models[name] = fitted

results_df = pd.DataFrame(model_results).sort_values(
    by=[
        "validation_macro_f1",
        "validation_balanced_accuracy",
        "validation_log_loss"
    ],
    ascending=[False, False, True]
).reset_index(drop=True)

display(results_df)


## 10. Select the best model

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_validation_model = fitted_models[best_model_name]

print("Selected model:", best_model_name)

validation_prediction = best_validation_model.predict(
    X_validation
)

print(
    classification_report(
        y_validation,
        validation_prediction,
        labels=[1, 2, 3, 4],
        target_names=[
            "Level 1",
            "Level 2",
            "Level 3",
            "Level 4"
        ],
        zero_division=0
    )
)


## 11. Refit and calibrate the selected model

Training and validation data are combined after model selection.

Calibration produces more meaningful probability estimates.

The test set remains untouched until final evaluation.


In [ ]:
combined_ml = pd.concat(
    [train_ml, validation_ml],
    ignore_index=True
)

X_combined = combined_ml[ALL_FEATURES]
y_combined = combined_ml["risk_level"].astype(int)

base_model = clone(
    candidate_models[best_model_name]
)

# Calibrate using internal stratified folds where possible.
minimum_class_count = int(
    y_combined.value_counts().min()
)

calibration_cv = min(5, minimum_class_count)

if calibration_cv < 2:
    raise ValueError(
        "Insufficient examples for probability calibration."
    )

calibrated_model = CalibratedClassifierCV(
    estimator=base_model,
    method="sigmoid",
    cv=calibration_cv
)

calibrated_model.fit(
    X_combined,
    y_combined
)

print(
    f"Calibrated {best_model_name} using "
    f"{calibration_cv}-fold cross-validation."
)


## 12. Evaluate the calibrated Levels 1–4 model on the fixed test set

In [ ]:
test_prediction = calibrated_model.predict(X_test)
test_probability = calibrated_model.predict_proba(X_test)

test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        test_prediction
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_test,
        test_prediction
    ),
    "macro_f1": f1_score(
        y_test,
        test_prediction,
        average="macro",
        zero_division=0
    ),
    "weighted_f1": f1_score(
        y_test,
        test_prediction,
        average="weighted",
        zero_division=0
    ),
    "log_loss": log_loss(
        y_test,
        test_probability,
        labels=[1, 2, 3, 4]
    )
}

try:
    test_metrics["macro_roc_auc"] = roc_auc_score(
        y_test,
        test_probability,
        multi_class="ovr",
        average="macro",
        labels=[1, 2, 3, 4]
    )
except Exception:
    test_metrics["macro_roc_auc"] = None

print(json.dumps(test_metrics, indent=2))

print(
    classification_report(
        y_test,
        test_prediction,
        labels=[1, 2, 3, 4],
        target_names=[
            "Level 1 — Very Low",
            "Level 2 — Low",
            "Level 3 — Moderate",
            "Level 4 — High"
        ],
        zero_division=0
    )
)


## 13. Display the confusion matrix

In [ ]:
conf_matrix = confusion_matrix(
    y_test,
    test_prediction,
    labels=[1, 2, 3, 4]
)

confusion_df = pd.DataFrame(
    conf_matrix,
    index=[
        "Actual L1",
        "Actual L2",
        "Actual L3",
        "Actual L4"
    ],
    columns=[
        "Predicted L1",
        "Predicted L2",
        "Predicted L3",
        "Predicted L4"
    ]
)

display(confusion_df)

plt.figure(figsize=(7, 6))
plt.imshow(conf_matrix)
plt.title("Test Confusion Matrix — Levels 1 to 4")
plt.xlabel("Predicted level")
plt.ylabel("Actual level")
plt.xticks(range(4), [1, 2, 3, 4])
plt.yticks(range(4), [1, 2, 3, 4])

for i in range(4):
    for j in range(4):
        plt.text(
            j,
            i,
            conf_matrix[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()


## 14. Measure ordinal prediction error

In [ ]:
absolute_error = np.abs(
    y_test.to_numpy() - test_prediction
)

ordinal_metrics = {
    "mean_absolute_level_error": float(
        absolute_error.mean()
    ),
    "within_one_level_accuracy": float(
        (absolute_error <= 1).mean()
    ),
    "exact_level_accuracy": float(
        (absolute_error == 0).mean()
    )
}

print(json.dumps(ordinal_metrics, indent=2))


## 15. Define the Level-5 critical-rule layer

These patterns are checked before accepting the Levels 1–4 model result.

The rule layer uses the original machine finding text and primary label.


In [ ]:
CRITICAL_LEVEL5_PATTERNS = {
    "Ventricular tachycardia": [
        r"\bventricular tachycardia\b"
    ],
    "Ventricular fibrillation": [
        r"\bventricular fibrillation\b"
    ],
    "Torsades pattern": [
        r"\btorsades\b"
    ],
    "Asystole": [
        r"\basystole\b"
    ],
    "Complete heart block": [
        r"\bcomplete (?:atrioventricular|av|heart) block\b",
        r"\bthird[- ]degree (?:atrioventricular|av) block\b"
    ],
    "Acute myocardial infarction pattern": [
        r"\bacute myocardial infarction\b",
        r"\bstemi\b"
    ]
}

def detect_level5_critical_rule(row):
    combined_text = " | ".join([
        str(row.get("machine_findings_text", "")),
        str(row.get("primary_label", "")),
        str(row.get("rhythm", ""))
    ])

    matched_reasons = []

    for reason, patterns in CRITICAL_LEVEL5_PATTERNS.items():
        for pattern in patterns:
            if re.search(
                pattern,
                combined_text,
                re.IGNORECASE
            ):
                matched_reasons.append(reason)
                break

    return list(dict.fromkeys(matched_reasons))


## 16. Evaluate Level-5 rule detection

In [ ]:
level5_evaluation_frames = [
    train_level5,
    validation_level5,
    test_level5
]

all_level5 = pd.concat(
    level5_evaluation_frames,
    ignore_index=True
)

if len(all_level5):
    level5_rule_results = []

    for _, row in all_level5.iterrows():
        matches = detect_level5_critical_rule(row)

        level5_rule_results.append({
            "source_file": row["source_file"],
            "primary_label": row.get("primary_label", ""),
            "machine_findings_text": row.get(
                "machine_findings_text",
                ""
            ),
            "rule_detected": bool(matches),
            "critical_reasons": " | ".join(matches)
        })

    level5_rule_df = pd.DataFrame(
        level5_rule_results
    )

    display(level5_rule_df)

    level5_rule_recall = float(
        level5_rule_df["rule_detected"].mean()
    )

    print(
        "Observed Level-5 rule recall:",
        round(level5_rule_recall, 4)
    )
else:
    level5_rule_df = pd.DataFrame()
    level5_rule_recall = None
    print("No Level-5 examples were available.")


## 17. Build the final five-level probability function

For normal machine-learning predictions:

- Levels 1–4 come from the calibrated classifier.
- Level 5 receives a small residual probability.

For critical-rule matches:

- Level 5 becomes the final result.
- The output clearly marks that Level 5 came from the rule layer.

The Level-5 value is a rule-based critical score, not a statistically calibrated probability.


In [ ]:
RISK_LEVEL_NAMES = {
    1: "Very Low",
    2: "Low",
    3: "Moderate",
    4: "High",
    5: "Very High"
}

def normalize_ml_probabilities_to_five(
    model_probabilities,
    residual_level5_probability=0.01
):
    model_probabilities = np.asarray(
        model_probabilities,
        dtype=float
    )

    model_probabilities = (
        model_probabilities
        / model_probabilities.sum()
    )

    scaled_first_four = (
        model_probabilities
        * (1 - residual_level5_probability)
    )

    return np.concatenate([
        scaled_first_four,
        [residual_level5_probability]
    ])

def critical_rule_distribution(
    ml_probabilities=None,
    critical_probability=0.90
):
    if ml_probabilities is None:
        remaining = np.array(
            [0.01, 0.02, 0.03, 0.04]
        )
    else:
        remaining = np.asarray(
            ml_probabilities,
            dtype=float
        )
        remaining = remaining / remaining.sum()

    remaining = remaining * (
        1 - critical_probability
    )

    return np.concatenate([
        remaining,
        [critical_probability]
    ])

def expected_risk_score(probabilities):
    return float(
        sum(
            level * probabilities[level - 1]
            for level in range(1, 6)
        )
    )


## 18. Final hybrid prediction function

In [ ]:
def predict_hybrid_risk(row):
    critical_reasons = detect_level5_critical_rule(row)

    feature_frame = pd.DataFrame(
        [{
            feature: row.get(feature, np.nan)
            for feature in ALL_FEATURES
        }]
    )

    feature_frame[CATEGORICAL_FEATURES] = (
        feature_frame[CATEGORICAL_FEATURES]
        .fillna("Unknown")
        .astype(str)
    )

    ml_probabilities = calibrated_model.predict_proba(
        feature_frame
    )[0]

    if critical_reasons:
        final_probabilities = critical_rule_distribution(
            ml_probabilities=ml_probabilities,
            critical_probability=0.90
        )

        predicted_level = 5
        prediction_source = "Critical rule layer"
        confidence_note = (
            "Level-5 value is a rule-based critical score, "
            "not a calibrated model probability."
        )
    else:
        final_probabilities = (
            normalize_ml_probabilities_to_five(
                ml_probabilities,
                residual_level5_probability=0.01
            )
        )

        predicted_level = int(
            np.argmax(final_probabilities) + 1
        )

        prediction_source = (
            "Calibrated machine-learning classifier"
        )

        confidence_note = (
            "Levels 1–4 are calibrated model probabilities; "
            "Level 5 is a small residual value."
        )

    return {
        "predicted_risk_level": predicted_level,
        "predicted_risk_name": (
            RISK_LEVEL_NAMES[predicted_level]
        ),
        "confidence": float(
            final_probabilities[predicted_level - 1]
        ),
        "expected_risk_score": expected_risk_score(
            final_probabilities
        ),
        "probability_level_1": float(
            final_probabilities[0]
        ),
        "probability_level_2": float(
            final_probabilities[1]
        ),
        "probability_level_3": float(
            final_probabilities[2]
        ),
        "probability_level_4": float(
            final_probabilities[3]
        ),
        "probability_level_5": float(
            final_probabilities[4]
        ),
        "prediction_source": prediction_source,
        "critical_rule_reasons": (
            " | ".join(critical_reasons)
        ),
        "confidence_note": confidence_note
    }


## 19. Test the hybrid system on the complete fixed test set

In [ ]:
hybrid_predictions = []

for _, row in test_df.iterrows():
    result = predict_hybrid_risk(row)

    hybrid_predictions.append({
        "source_file": row["source_file"],
        "actual_risk_level": int(
            row["risk_level"]
        ),
        "actual_risk_name": row["risk_name"],
        "possible_ecg_findings": row.get(
            "possible_ecg_findings",
            ""
        ),
        "risk_reasons": row.get(
            "risk_reasons",
            ""
        ),
        **result
    })

hybrid_test_results = pd.DataFrame(
    hybrid_predictions
)

display(hybrid_test_results.head(20))


## 20. Evaluate complete five-level hybrid predictions

In [ ]:
hybrid_y_true = hybrid_test_results[
    "actual_risk_level"
].astype(int)

hybrid_y_pred = hybrid_test_results[
    "predicted_risk_level"
].astype(int)

hybrid_metrics = {
    "five_level_accuracy": accuracy_score(
        hybrid_y_true,
        hybrid_y_pred
    ),
    "five_level_balanced_accuracy": (
        balanced_accuracy_score(
            hybrid_y_true,
            hybrid_y_pred
        )
    ),
    "five_level_macro_f1": f1_score(
        hybrid_y_true,
        hybrid_y_pred,
        average="macro",
        zero_division=0
    ),
    "five_level_weighted_f1": f1_score(
        hybrid_y_true,
        hybrid_y_pred,
        average="weighted",
        zero_division=0
    ),
    "mean_absolute_level_error": float(
        np.abs(
            hybrid_y_true.to_numpy()
            - hybrid_y_pred.to_numpy()
        ).mean()
    ),
    "within_one_level_accuracy": float(
        (
            np.abs(
                hybrid_y_true.to_numpy()
                - hybrid_y_pred.to_numpy()
            ) <= 1
        ).mean()
    )
}

print(json.dumps(hybrid_metrics, indent=2))

print(
    classification_report(
        hybrid_y_true,
        hybrid_y_pred,
        labels=[1, 2, 3, 4, 5],
        target_names=[
            "Level 1 — Very Low",
            "Level 2 — Low",
            "Level 3 — Moderate",
            "Level 4 — High",
            "Level 5 — Very High"
        ],
        zero_division=0
    )
)


## 21. Display the five-level confusion matrix

In [ ]:
hybrid_confusion_matrix = confusion_matrix(
    hybrid_y_true,
    hybrid_y_pred,
    labels=[1, 2, 3, 4, 5]
)

hybrid_confusion_df = pd.DataFrame(
    hybrid_confusion_matrix,
    index=[
        "Actual L1",
        "Actual L2",
        "Actual L3",
        "Actual L4",
        "Actual L5"
    ],
    columns=[
        "Predicted L1",
        "Predicted L2",
        "Predicted L3",
        "Predicted L4",
        "Predicted L5"
    ]
)

display(hybrid_confusion_df)

plt.figure(figsize=(8, 7))
plt.imshow(hybrid_confusion_matrix)
plt.title("Hybrid Five-Level Test Confusion Matrix")
plt.xlabel("Predicted level")
plt.ylabel("Actual level")
plt.xticks(range(5), [1, 2, 3, 4, 5])
plt.yticks(range(5), [1, 2, 3, 4, 5])

for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            hybrid_confusion_matrix[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()


## 22. Preview the final application output

In [ ]:
def display_final_prediction(row):
    result = predict_hybrid_risk(row)

    print("ECG Risk Assessment")
    print("===================")
    print(
        f"Predicted Risk: Level "
        f"{result['predicted_risk_level']} — "
        f"{result['predicted_risk_name']}"
    )
    print(
        f"Confidence / critical score: "
        f"{100 * result['confidence']:.1f}%"
    )
    print(
        f"Expected risk score: "
        f"{result['expected_risk_score']:.2f}/5"
    )

    print("\nRisk probabilities:")
    for level in range(1, 6):
        value = result[
            f"probability_level_{level}"
        ]
        print(
            f"  Level {level} — "
            f"{RISK_LEVEL_NAMES[level]}: "
            f"{100 * value:.1f}%"
        )

    print("\nPossible ECG findings:")
    finding_text = str(
        row.get(
            "possible_ecg_findings",
            "Not available"
        )
    )
    for finding in finding_text.split("|"):
        finding = finding.strip()
        if finding:
            print(" •", finding)

    print("\nReasons:")
    if result["critical_rule_reasons"]:
        for reason in result[
            "critical_rule_reasons"
        ].split("|"):
            print(" •", reason.strip())
    else:
        reason_text = str(
            row.get(
                "risk_reasons",
                "Model-based assessment"
            )
        )
        for reason in reason_text.split("|"):
            reason = reason.strip()
            if reason:
                print(" •", reason)

    print("\nPrediction source:")
    print(" ", result["prediction_source"])

    print("\nProbability note:")
    print(" ", result["confidence_note"])

    print(
        "\nDisclaimer: ECG-based screening output only; "
        "not a confirmed diagnosis."
    )

sample = test_df.sample(
    1,
    random_state=42
).iloc[0]

display_final_prediction(sample)


## 23. Save the final model and metadata

In [ ]:
output_dir = Path(
    "/content/ecg_hybrid_model_outputs"
)

if output_dir.exists():
    shutil.rmtree(output_dir)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    calibrated_model,
    output_dir / "ecg_risk_model_levels_1_to_4.pkl"
)

joblib.dump(
    {
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "all_features": ALL_FEATURES,
        "risk_level_names": RISK_LEVEL_NAMES,
        "critical_level5_patterns": (
            CRITICAL_LEVEL5_PATTERNS
        ),
        "selected_model": best_model_name
    },
    output_dir / "ecg_hybrid_model_config.pkl"
)

results_df.to_csv(
    output_dir / "model_comparison.csv",
    index=False
)

confusion_df.to_csv(
    output_dir / "levels_1_to_4_confusion_matrix.csv"
)

hybrid_confusion_df.to_csv(
    output_dir / "hybrid_five_level_confusion_matrix.csv"
)

hybrid_test_results.to_csv(
    output_dir / "hybrid_test_predictions.csv",
    index=False
)

if len(level5_rule_df):
    level5_rule_df.to_csv(
        output_dir / "level5_rule_evaluation.csv",
        index=False
    )

evaluation_summary = {
    "selected_model": best_model_name,
    "training_levels": [1, 2, 3, 4],
    "critical_rule_level": 5,
    "training_rows_levels_1_to_4": int(
        len(combined_ml)
    ),
    "fixed_test_rows_levels_1_to_4": int(
        len(test_ml)
    ),
    "fixed_test_rows_all_levels": int(
        len(test_df)
    ),
    "level5_examples_total": int(
        len(all_level5)
    ),
    "level5_rule_recall_observed": (
        level5_rule_recall
    ),
    "levels_1_to_4_test_metrics": test_metrics,
    "levels_1_to_4_ordinal_metrics": (
        ordinal_metrics
    ),
    "hybrid_five_level_test_metrics": (
        hybrid_metrics
    ),
    "probability_note": (
        "Levels 1–4 use calibrated model probabilities. "
        "Level 5 uses a rule-based critical score."
    ),
    "clinical_note": (
        "The model is weakly supervised using provisional "
        "rule-generated ECG screening labels and requires "
        "clinical validation."
    )
}

with open(
    output_dir / "evaluation_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        evaluation_summary,
        file,
        indent=2,
        ensure_ascii=False
    )

archive_path = shutil.make_archive(
    "/content/ecg_hybrid_model_outputs",
    "zip",
    output_dir
)

print(
    json.dumps(
        evaluation_summary,
        indent=2
    )
)

print("\nCreated:", archive_path)


## 24. Download the final model package

In [ ]:
from google.colab import files

files.download(
    "/content/ecg_hybrid_model_outputs.zip"
)


## Final files

The ZIP contains:

- `ecg_risk_model_levels_1_to_4.pkl`
- `ecg_hybrid_model_config.pkl`
- `model_comparison.csv`
- `levels_1_to_4_confusion_matrix.csv`
- `hybrid_five_level_confusion_matrix.csv`
- `hybrid_test_predictions.csv`
- `level5_rule_evaluation.csv`
- `evaluation_summary.json`

Share `evaluation_summary.json` and `model_comparison.csv` after running the notebook so the final model quality can be reviewed.
